### Check Python version

In [4]:
from platform import python_version_tuple

assert (
    python_version_tuple()[:2] == ("3", "11")
), "You are not running Python 3.11. Make sure to run Python through the course Conda environment."

In [5]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm
import tifffile as tiff

from convert_labels import from_rgb_labels_to_int_labels


# EXTRACTION OF IMAGES STARTS HERE

- from each original image tile (6000 x 6000 x 3)
    - we will extract patches of size (512 x 512 x 3)
    - `stride = 512` therefore, the no overlap between neighboring patches


- In order to have NO overlap, such that no parts of original image tiles will be used more than others during training, and also in order to reduce the total number of training images.  
    - The rightmost part and bottom part from the original image tiles will **not** be extracted.
    - 512*11 = 5632
    - i.e. (5632 x 5632) pixels will be extracted from each (6000 x 6000) image tile.
    - This will result in 11*11=121 images of dimension (512 x 512) to be extracted for each original image tile

- For training data
    - If 95% or more of an extracted (512 x 512) image consists of a single class, this will  **not** be saved into training data.

In [6]:
# =====================================================
# CONFIGURATION
# =====================================================

IMAGE_DIR = Path("potsdam/Ortho_RGB_train")
LABEL_DIR = Path("potsdam/Labels_all")

OUT_IMAGE_DIR = Path("train_patches/images")
OUT_LABEL_DIR = Path("train_patches/labels")

PATCH_SIZE = 512
STRIDE = 512  # 512 = no overlap, 256 = 50% overlap

OUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUT_LABEL_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# =====================================================
# EXTRACTION of Train patches
# =====================================================

image_files = sorted(IMAGE_DIR.glob("*.tif"))


for image_path in tqdm(image_files):

    # Labels have same name as image with ending "_RGB" replaced with "_label")
    label_path = LABEL_DIR / image_path.name.replace("_RGB", "_label")

    if not label_path.exists():
        print(f"Missing label for {image_path.name}")
        continue

    image = tiff.imread(image_path)
    label = tiff.imread(label_path)

    # shape of image read using tifffile is (H, W, C) 
    # for Potsdam images, this is== (6000, 6000, 3) 
    h, w = image.shape[0], image.shape[1]


    stem = (image_path.stem).replace("_RGB", "")

    patch_count = 0
    skip_count = 0

    # Calculate the x and y starting coordinates of the patches
    xs = list(range(0, w - PATCH_SIZE, STRIDE))
    ys = list(range(0, h - PATCH_SIZE, STRIDE))


    
    for y in ys:
        for x in xs:

            image_patch = image[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE ]

            label_patch = label[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE ]


            # create a new label patch with integer labels {0,1,2,3,4,5} instead of RGB values
            label_patch_int = from_rgb_labels_to_int_labels(label_patch)

            # Calculate the fraction of the patch that belongs to the majority class
            majority_fraction = np.max(np.bincount(label_patch_int.flatten())) / label_patch_int.size

            # If the majority class occupies more than 95% of the patch, skip this patch
            # i.e. if the patch is highly uniform, we skip it
            if majority_fraction > 0.95:
                skip_count += 1
                continue

            
            patch_name = f"{stem}_{y}_{x}.tif"

            tiff.imwrite(
                OUT_IMAGE_DIR / patch_name,
                image_patch )

            tiff.imwrite(
                OUT_LABEL_DIR / patch_name,
                label_patch )

            patch_count += 1

    print(f"From image tile {image_path.name}: {patch_count} patches extracted, {skip_count} patches skipped due to majority class > 95%")

print("Finished.")

  0%|          | 0/24 [00:00<?, ?it/s]

  4%|▍         | 1/24 [00:07<02:43,  7.12s/it]

From image tile top_potsdam_2_10_RGB.tif: 116 patches extracted, 5 patches skipped due to majority class > 95%


  8%|▊         | 2/24 [00:14<02:42,  7.39s/it]

From image tile top_potsdam_2_11_RGB.tif: 114 patches extracted, 7 patches skipped due to majority class > 95%


...

From image tile top_potsdam_7_7_RGB.tif: 121 patches extracted, 0 patches skipped due to majority class > 95%


 96%|█████████▌| 23/24 [03:16<00:11, 11.44s/it]

From image tile top_potsdam_7_8_RGB.tif: 121 patches extracted, 0 patches skipped due to majority class > 95%


100%|██████████| 24/24 [03:23<00:00,  8.48s/it]

From image tile top_potsdam_7_9_RGB.tif: 118 patches extracted, 3 patches skipped due to majority class > 95%
Finished.


In [8]:
class_to_category = {
        0 :  "Impervious surfaces",
        1 :  "Building",
        2 :  "Low vegetation",
        3 :  "Tree",
        4 :  "Car",
        5 :  "Clutter/background"
    }

In [9]:
# Initialize counts for classes 0-5
counts = np.zeros(6, dtype=int)  

counter = 0

for image_path in tqdm(image_files):

    label_path = LABEL_DIR / image_path.name.replace("_RGB", "_label")
    label = tiff.imread(label_path)
    
    # create a new label patch with integer labels {0,1,2,3,4,5} instead of RGB values
    label_int = from_rgb_labels_to_int_labels(label)

    # Calculate the occurrence of each class in the label image
    a = np.bincount(label_int.flatten())

    counts += a  # Update counts for each class
    counter += 1


print("Counts of classes in the training dataset")
print()

for i in range(6):
    print(f"Class {i} ({class_to_category[i]}): {counts[i]} ({counts[i] / counts.sum() * 100 : 2.2f}% of total)")

100%|██████████| 24/24 [01:24<00:00,  3.54s/it]

Counts of classes in the training dataset

Class 0 (Impervious surfaces): 245930445 ( 28.46% of total)
Class 1 (Building): 230875852 ( 26.72% of total)
Class 2 (Low vegetation): 203358663 ( 23.54% of total)
Class 3 (Tree): 126352970 ( 14.62% of total)
Class 4 (Car): 14351363 ( 1.66% of total)
Class 5 (Clutter/background): 43130707 ( 4.99% of total)
